In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from utils.helper import BatchControlHelper
from pyspark.sql import DataFrame
from pyspark.sql import functions as f
import json

In [0]:
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text(
    "table_metadata",
     "{'table_id': '2', 'table_name': 'customer_history', 'source_system': 'sqlserver', 'source_schema': 'data_source', 'source_table': 'customer_history', 'source_path': '', 'target_layer': 'bronze', 'bronze_schema': 'bronze', 'silver_schema': 'silver', 'gold_schema': '', 'active_flag': 'True', 'load_order': '2', 'created_at': '2026-10-03 09:36:35.783849'}"
)
dbutils.widgets.text(
    "table_config",
    "{'load_type': 'MERGE', 'primary_key': 'customer_id,effective_from_date', 'watermark_column': 'source_modified_at'}"
)
dbutils.widgets.text("run_id","123")
dbutils.widgets.text("processing_date", "2025-01-02", "As-of date used when generating the initial load (YYYY-MM-DD)")
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"], "Reprocess tables even if already SUCCESS")

processing_date = dbutils.widgets.get("processing_date")
force_reload = dbutils.widgets.get("force_reload") == "true"
run_id=dbutils.widgets.get("run_id")
env = dbutils.widgets.get("environment")
table_metadata = json.loads(dbutils.widgets.get("table_metadata").replace("'", '"'))
table_config = json.loads(dbutils.widgets.get("table_config").replace("'", '"'))

batch_id = f"initial-{processing_date.replace('-', '')}"

config = {
    "source_table": table_metadata["source_table"],
    "source_schema": table_metadata["source_schema"],
    "target_table": table_metadata["table_name"],
    "target_layer": table_metadata["target_layer"],
    "load_type": table_config["load_type"],
    "primary_key": table_config["primary_key"].split(','),
    "watermark_column": table_config["watermark_column"],
    'processing_date': processing_date,
    'force_reload': force_reload,
    'batch_id': batch_id,
    'run_id': run_id,
    'env': env
}

config

In [0]:
batch_controller = BatchControlHelper(
    spark=spark, 
    batch_control_table=f"{config['env']}_banking.metadata.batch_control",
    watermark_table=f"{config['env']}_banking.metadata.table_watermarks"
)

In [0]:
def ingest_source_table(config: dict) -> dict:
    source_table_name = f"{config['env']}_banking.{config['source_schema']}.{config['source_table']}"
    full_table_name = f"{config['env']}_banking.{config['target_layer']}.{config['target_table']}"

    if not force_reload and batch_controller.has_succeeded(config['batch_id'], config['source_table']):
        print(f"SKIP    {config['source_table']}: already SUCCESS for batch {config['batch_id']}")
        return {"source_table": config['source_table'], "status": "SKIPPED", "record_count": None}
    
    run_id = batch_controller.start_batch(config['batch_id'], config['source_table'], config['target_table'], config['target_layer'], config['load_type'])
    print("RUN ID : ", run_id)
    try:
        previous_watermark = batch_controller.get_watermark(config['target_table'], config['target_layer']) # '1999-01-01'        
        source_data = spark.read.table(source_table_name)
        source_max_timestamp = (
            source_data.select(f.max(f.col(config['watermark_column']))).first()[0] if config['watermark_column'] is not None else None  # '2024-12-31'
        )
        if not force_reload:
            source_data = source_data.filter(f.col(config['watermark_column']) > f.lit(previous_watermark))
        
        
        source_data = source_data.withColumn("processing_date", f.lit(config['processing_date']))\
                .withColumn("_ingestion_batch_id", f.lit(config['batch_id']))\
                .withColumn("_ingestion_run_id", f.lit(run_id))        
        
        null_key_count = source_data.filter(
            f.greatest(*[f.col(pk).isNull().cast("int") for pk in config['primary_key']]) == 1).count()
        
        if null_key_count > 0:
            raise ValueError(f"{null_key_count} row(s) have a null value in primary key columns {config['primary_key']}")

        record_count = source_data.count()
        if record_count > 0:
            source_data.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(full_table_name)
        
        batch_controller.update_watermark(config['target_layer'], config['target_table'], source_max_timestamp, run_id)
        batch_controller.complete_batch(run_id, record_count)
        print(
            f"SUCCESS {source_table_name} -> {full_table_name} "
            f"({record_count} new rows; watermark={config['watermark_column']})"
        )
        return {"source_table": config['source_table'], "status": "SUCCESS", "record_count": record_count}

    except Exception as error:
        batch_controller.fail_batch(run_id, str(error))
        print(f"FAILED  {config['source_table']}: {error}")
        raise Exception(error)

In [0]:
result = ingest_source_table(config)

In [0]:
result